# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [13]:
import os
import numpy as np
import importlib.util
import pickle
from scipy.sparse import coo_matrix
from spektral.data import Graph


def load_params(py_path):
    """
    Dynamically loads a Python file containing parameters, e.g.:
        v0 = [0.1, 1.3]
        W = [[0.0, 0.08],
             [0.08, 0.0]]
        A0 = [0.9, 0.9]
        P0 = [3.812, 3.812]
        Dr = 50
        kappa_A = 0.4
        kappa_P = 0.07
        a = 0.25
        k = 2.5
    """
    print(f"> Attempting to load parameters from: {py_path}")
    spec = importlib.util.spec_from_file_location("param_module", py_path)
    param_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(param_module)
    
    # Gather the parameters into a dictionary:
    p = {
        "v0": getattr(param_module, "v0", None),
        "W": getattr(param_module, "W", None),
        "A0": getattr(param_module, "A0", None),
        "P0": getattr(param_module, "P0", None),
        "Dr": getattr(param_module, "Dr", None),
        "kappa_A": getattr(param_module, "kappa_A", None),
        "kappa_P": getattr(param_module, "kappa_P", None),
        "a": getattr(param_module, "a", None),
        "k": getattr(param_module, "k", None),
    }
    print(f"> Loaded parameters: {p}")
    return p


def build_dataset(
    data_dir: str,
    param_path: str,
    t_start: int = 0,
    t_end: int = 200,
    t_step: int = 1,
):
    """
    Build a dataset of Graph snapshots (G_t) from time t in [t_start, t_end],
    stepping by t_step. For each t, we also produce the adjacency at t+1 (if 
    available) for training a GNN that predicts adjacency changes from t -> t+1.

    Key difference: we only gather "unique" edges (i<j), then duplicate them 
    to produce 2E edges in a well-defined forward-block + reverse-block.

    Returns:
      A list of (Graph, adjacency_{t+1}) for times t.
    """
    print(f"\n=== build_dataset ===")
    print(f"Data dir  = {data_dir}")
    print(f"Param file= {param_path}")
    print(f"Time range= [{t_start}, {t_end}] with step={t_step}\n")

    # 1) Load the param file
    p = load_params(param_path)
    
    # 2) We'll store a list of (G_t, adjacency_{t+1}) for each valid t
    dataset = []

    # 3) Iterate over time steps
    t_list = range(t_start, t_end + 1, t_step)
    print(f">> Will look for data_{{t}}.npy from t in {list(t_list)}")

    for t in t_list:
        data_file_t = os.path.join(data_dir, f"data_{t}.npy")
        if not os.path.exists(data_file_t):
            print(f" - data_{t}.npy not found. Skipping.")
            continue

        print(f"\nLoading timepoint t={t} from {data_file_t}")
        data_t = np.load(data_file_t, allow_pickle=True).item()

        # We'll only build a label adjacency if (t + t_step) also exists
        t_next = t + t_step
        data_file_tplus = os.path.join(data_dir, f"data_{t_next}.npy")
        if os.path.exists(data_file_tplus):
            data_tplus = np.load(data_file_tplus, allow_pickle=True).item()
            adj_label = data_tplus["cell_adj"]  # shape [n_c, n_c]
            print(f"   Found adjacency label at t+{t_step} = {t_next}")
        else:
            adj_label = None
            print(f"   No data for t+{t_step} = {t_next}. Label is None.")

        # Extract your raw data from dictionary:
        cell_x    = data_t["cell_x"]           # shape [n_c, 2]
        cell_type = data_t["cell_type"]        # shape [n_c]
        area      = data_t["area"]             # shape [n_c]
        perimeter = data_t["perimeter"]        # shape [n_c]
        cell_adj  = data_t["cell_adj"]         # shape [n_c, n_c]

        # "edge_voronoi_length" might or might not exist
        voronoi_len = data_t.get("edge_voronoi_length", None)
        if voronoi_len is None:
            print("   edge_voronoi_length not found in data dict.")
        else:
            print("   Found edge_voronoi_length in data dict.")

        n_c = cell_x.shape[0]
        print(f"   n_c = {n_c} cells in this timepoint")

        # 4) Build node features for time t
        #    Example: [ x_i, y_i, area_i, perimeter_i, cell_type_i, 
        #               A0(type), P0(type), Dr, kappa_A, kappa_P, a, k ]
        node_feats = []
        for i in range(n_c):
            ctype = cell_type[i]
            A0_val = p["A0"][ctype]
            P0_val = p["P0"][ctype]
            (x_i, y_i) = cell_x[i]
            feats = [
                x_i,
                y_i,
                area[i],
                perimeter[i],
                float(ctype),
                A0_val,
                P0_val,
                p["Dr"],
                p["kappa_A"],
                p["kappa_P"],
                p["a"],
                p["k"],
            ]
            node_feats.append(feats)
        node_feats = np.array(node_feats, dtype=np.float32)  # shape [n_c, D]

        # 5) Build adjacency + edge features, but first we gather only "unique" edges (i<j)
        unique_rows = []
        unique_cols = []
        unique_edge_feats = []

        for i in range(n_c):
            for j in range(i+1, n_c):
                if cell_adj[i, j] == 1:
                    # i < j, this is the unique edge
                    unique_rows.append(i)
                    unique_cols.append(j)
                    
                    # Voronoi length if available
                    vlen = 0.0
                    if voronoi_len is not None:
                        vlen = voronoi_len[i, j]
                    
                    # W_interaction for this pair
                    ctype_i = cell_type[i]
                    ctype_j = cell_type[j]
                    Wij = p["W"][ctype_i][ctype_j]
                    
                    # Build the edge feature for i->j
                    unique_edge_feats.append([vlen, Wij])

        E_unique = len(unique_rows)  # number of unique edges

        # 6) Duplicate them to get forward + reverse
        rows_duplicated = unique_rows + unique_cols  # length 2*E_unique
        cols_duplicated = unique_cols + unique_rows  # same length
        edge_feats_duplicated = unique_edge_feats + unique_edge_feats  # also length 2*E_unique

        row_ar = np.array(rows_duplicated, dtype=np.int32)
        col_ar = np.array(cols_duplicated, dtype=np.int32)
        edge_feats_ar = np.array(edge_feats_duplicated, dtype=np.float32)

        data_ar = np.ones_like(row_ar, dtype=np.float32)  # or whatever weight you want
        a_coo = coo_matrix((data_ar, (row_ar, col_ar)), shape=(n_c, n_c))

        # 7) Construct the Spektral Graph for time t
        #    shape of edge_feats_ar => [2*E_unique, D_edge]
        G_t = Graph(
            x=node_feats,
            a=a_coo,
            e=edge_feats_ar
        )
        
        # 8) Append (G_t, adj_label) to dataset
        dataset.append((G_t, adj_label))

    print(f"\nBuild complete. Total {len(dataset)} snapshots processed.")
    return dataset

In [14]:

if __name__ == "__main__":
    # Example usage with your lists of data_dirs and param_files.
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_{i}"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    # We'll define some time config:
    T_START = 0
    T_END = 2000   # up to 2000
    T_STEP = 200     # maybe your data are spaced in increments of 4

    # We'll build dataset for each sim ID in turn
    all_datasets = []
    for sim_idx in range(10):
        ddir = data_dirs[sim_idx]
        pfile = param_files[sim_idx]
        ds = build_dataset(
            data_dir=ddir,
            param_path=pfile,
            t_start=T_START,
            t_end=T_END,
            t_step=T_STEP
        )
        print(
            f"Simulation ID = {sim_idx+1}: built dataset with {len(ds)} steps "
            f"from directory '{ddir}'."
        )
        all_datasets.append(ds)

    # Now save `all_datasets` to a local file (e.g., "all_datasets.pkl"):

    save_path = "all_datasets.pkl"
    print(f"\nSaving all_datasets to {save_path} ...")
    with open(save_path, "wb") as f:
        pickle.dump(all_datasets, f)

    print("Done. The file contains a list of 10 datasets (one per simulation).")

    # Now all_datasets is a list of datasets, each one being a list of (Graph, adj_label).
    # all_datasets[i] corresponds to simulation i+1.
    
    # Example: just checking the first step from simulation #1
    if len(all_datasets[0]) > 0:
        graph_0, adj_label_0 = all_datasets[0][0]
        print("First Graph node features shape:", graph_0.x.shape)
        print("First Graph adjacency shape:", graph_0.a.shape)
        print("First Graph edge feature shape:", graph_0.e.shape)
        if adj_label_0 is not None:
            print("Next adjacency shape:", adj_label_0.shape)



=== build_dataset ===
Data dir  = /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_1
Param file= /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
Time range= [0, 2000] with step=200

> Attempting to load parameters from: /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/1.py
> Loaded parameters: {'v0': [0.1, 1.3], 'W': ([0.0, 0.08], [0.08, 0.0]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 50, 'kappa_A': 0.4, 'kappa_P': 0.07, 'a': 0.25, 'k': 2.5}
>> Will look for data_{t}.npy from t in [0, 200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800, 2000]

Loading timepoint t=0 from /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I/matrix_output_1/data_0.npy
   Found adjacency label at t+200 = 200
   Found edge_voronoi_length in data dict.
   n_c = 840 cells in this timepoint

Loading timepoint t=200 from /Users/sophia01px2019/Downloads/gartner_lab_rotati

Here is where we:

Load all_datasets.pkl.
For each (Graph_t, adj_label_{t+1}), convert to PyG format.
Use NeighborLoader to sample subgraphs for each node (or a fraction of them).
Build next-step labels (adj, area, perimeter, boundary length, etc.) restricted to subgraph nodes.
Save subgraph+label to some file for training.

In [22]:
# Feature layouts: 

# node_feats[i] = [
#     x_i,       # index 0
#     y_i,       # index 1
#     area[i],   # index 2
#     perimeter[i], # 3
#     cell_type[i], # 4
#     A0_val,    # 5
#     P0_val,    # 6
#     Dr,        # 7
#     kappa_A,   # 8
#     kappa_P,   # 9
#     a,         # 10
#     k,         # 11
# ]
# edge_feats[e] = [
#     exist, # index 0 => adjacency 
#     vlen,  # index 1 => edge_voronoi_length
#     Wij,   # index 2 => W_interaction
# ]

In [3]:
import torch_scatter
import torch_sparse
import torch_cluster
import torch_spline_conv
import torch_geometric

In [3]:
import os
import pickle
import numpy as np
import torch
from torch_geometric.data import Data

def convert_spektral_to_pyg(spektral_graph):
    """
    Convert a spektral Graph object to PyG Data, ensuring edge_attr has shape [E,3].
    Indices for each edge:
       0 => vlen(t)
       1 => W
       2 => exists(t)  (1 if it's in spektral_graph, 0 otherwise)
    """

    x = torch.FloatTensor(spektral_graph.x)  # shape [N, 12] (node feats)

    # Convert adjacency from spektral => PyG edge_index
    row = spektral_graph.a.row
    col = spektral_graph.a.col
    edge_index_np = np.vstack([row, col])  # shape [2, E]
    edge_index_t = torch.LongTensor(edge_index_np)

    # Original edge features: shape [E,2], or None
    # We'll produce a new array [E,3] => [vlen, W, exists(t)].
    if spektral_graph.e is not None:
        old_e = spektral_graph.e  # shape [E,2]
        E = old_e.shape[0]
        new_e = np.zeros((E, 3), dtype=np.float32)
        new_e[:,0:2] = old_e[:,0:2]  # copy vlen, W
        new_e[:,2] = 1.0            # since these edges exist(t) in Spektral => 1
        edge_attr = torch.FloatTensor(new_e)
    else:
        # If there's no edge feature array, build one => all edges have vlen=0, W=0, exist=1
        E = edge_index_np.shape[1]
        new_e = np.zeros((E,3), dtype=np.float32)
        new_e[:,2] = 1.0
        edge_attr = torch.FloatTensor(new_e)

    pyg_data = Data(
        x=x,                   # shape [N, 12]
        edge_index=edge_index_t,   # shape [2, E]
        edge_attr=edge_attr,   # shape [E, 3]
    )
    return pyg_data

def build_fullgraph_label(
    pyg_data_t,
    adjacency_tplus,   # shape [N, N], adjacency at t+1
    x_time_t,          # shape [N, 12], node feats at t
    x_time_tplus,      # shape [N, 12], node feats at t+1
    e_time_t=None,     # shape [E2, 2] => old [vlen, W] at time t
    e_time_tplus=None, # shape [E2, 2] => old [vlen, W] at t+1
    row_t=None, col_t=None,
    row_tplus=None, col_tplus=None
):
    """
    Build a label dict for the entire graph at time t => next-step info for t+1.
    We'll produce:
      node_feats => shape [N,12] (the next-step node features)
      edge_feats => shape [E,3]  => [vlen(t+1), W, exists(t+1)]
    to match the GNN output dimension.
    """
    N = pyg_data_t.num_nodes
    E = pyg_data_t.num_edges

    # 1) node_feats => shape [N,12], just copy x_time_tplus
    #    The GNN will do partial MSE on dynamic dims [0..3], no penalty on 4..11
    node_feats = x_time_tplus  # shape [N,12]

    # 2) Build edge_feats => shape [E,3]
    #    we define columns:
    #      0 => vlen(t+1)
    #      1 => W (constant)
    #      2 => exists(t+1) => from adjacency_tplus
    edge_feats = np.zeros((E,3), dtype=np.float32)

    # (a) First fill W from e_time_t (if we want the same W)
    # We can replicate the same logic from row_t / col_t
    # We'll build a map from (min(r,c)) => e_time_tplus for vlen(t+1)
    # Then we also build from e_time_t => get W
    # But note we might also read W from the CURRENT PyG edge_attr? 
    # => that might be simpler, but let's do consistent with row_t, col_t

    # Build a map for time t => (r,c) => W
    w_map_t = {}
    if e_time_t is not None and row_t is not None:
        E_sp = len(row_t)
        for i in range(E_sp):
            r = row_t[i]
            c = col_t[i]
            if r>c: r,c = c,r
            # old: e_time_t[i,0] => vlen, e_time_t[i,1] => W
            w_map_t[(r,c)] = e_time_t[i,1]

    # Build a map for time t+1 => (r,c) => vlen(t+1)
    vlen_map_tp1 = {}
    if e_time_tplus is not None and row_tplus is not None:
        E_sp2 = len(row_tplus)
        for i in range(E_sp2):
            r = row_tplus[i]
            c = col_tplus[i]
            if r>c: r,c = c,r
            # e_time_tplus[i,0] => vlen(t+1), e_time_tplus[i,1] => W
            vlen_map_tp1[(r,c)] = e_time_tplus[i,0]

    # Now fill edge_feats by iterating over PyG edges
    ei_src = pyg_data_t.edge_index[0]
    ei_dst = pyg_data_t.edge_index[1]
    for e_i in range(E):
        s = ei_src[e_i].item()
        d = ei_dst[e_i].item()
        s_, d_ = (s,d) if s<d else (d,s)

        # vlen(t+1)
        if (s_,d_) in vlen_map_tp1:
            edge_feats[e_i,0] = vlen_map_tp1[(s_,d_)]
        else:
            edge_feats[e_i,0] = 0.0

        # W
        if (s_,d_) in w_map_t:
            edge_feats[e_i,1] = w_map_t[(s_,d_)]
        else:
            edge_feats[e_i,1] = 0.0

        # exists(t+1)
        if adjacency_tplus[s,d] == 1:
            edge_feats[e_i,2] = 1.0
        else:
            edge_feats[e_i,2] = 0.0

    # We'll return a dictionary containing exactly these arrays:
    label_dict = {
        # The GNN will do partial MSE on dynamic dims of node_feats => indices [0..3]
        # plus partial MSE on edge_feats => index [0], BCE on edge_feats => index [2], ignoring [1].
        "node_feats": node_feats,   # shape [N,12]
        "edge_feats": edge_feats,   # shape [E,3]
    }
    return label_dict

import os
import pickle
import numpy as np
import torch
from torch_geometric.data import Data

def convert_spektral_to_pyg(spektral_graph):
    """
    Convert a spektral Graph object to PyG Data, producing edge_attr of shape [E,3].
    Indices for each edge: [vlen(t), W, exists(t)].
    We set 'exists(t)=1' for all edges that appear in spektral_graph.
    """
    x = torch.FloatTensor(spektral_graph.x)  # shape [N, 12]

    # Spektral adjacency => PyG edge_index
    row = spektral_graph.a.row
    col = spektral_graph.a.col
    edge_index_np = np.vstack([row, col])  # shape [2, E]
    edge_index_t = torch.LongTensor(edge_index_np)

    # Original e: shape [E,2] => [vlen, W]
    # We'll produce [E,3] => [vlen, W, exists(t)=1].
    if spektral_graph.e is not None:
        old_e = spektral_graph.e
        E = old_e.shape[0]
        new_e = np.zeros((E,3), dtype=np.float32)
        new_e[:,0:2] = old_e[:,0:2]  # copy vlen, W
        new_e[:,2] = 1.0            # since these edges exist at t
        edge_attr = torch.FloatTensor(new_e)
    else:
        E = edge_index_np.shape[1]
        new_e = np.zeros((E,3), dtype=np.float32)
        new_e[:,2] = 1.0
        edge_attr = torch.FloatTensor(new_e)

    pyg_data = Data(
        x=x,  # [N,12]
        edge_index=edge_index_t,
        edge_attr=edge_attr # [E,3]
    )
    return pyg_data

def build_fullgraph_label(
    pyg_data_t,
    adjacency_tplus,   # shape [N, N], adjacency at t+1
    x_time_t,          # shape [N,12], node feats at t
    x_time_tplus,      # shape [N,12], node feats at t+1
    e_time_t=None,     # shape [E2,2] => old [vlen, W] at t
    e_time_tplus=None, # shape [E2,2] => old [vlen, W] at t+1
    row_t=None, col_t=None,
    row_tplus=None, col_tplus=None
):
    """
    Builds label_dict with:
      'node_feats': shape [N,12] => next-step node feats
      'edge_feats': shape [E,3] => [vlen(t+1), W, exists(t+1)].
    The GNN will do partial MSE on node[0..3], edge[0], plus BCE on edge[2].
    """
    N = pyg_data_t.num_nodes
    E = pyg_data_t.num_edges

    # node_feats => shape [N,12]
    node_feats = x_time_tplus

    # Build edge_feats => shape [E,3]:
    #   0 => vlen(t+1)
    #   1 => W
    #   2 => exists(t+1)

    edge_feats = np.zeros((E,3), dtype=np.float32)

    # Maps for time t => W, time t+1 => vlen(t+1)
    w_map_t = {}
    if e_time_t is not None and row_t is not None:
        E_sp = len(row_t)
        for i in range(E_sp):
            r = row_t[i]
            c = col_t[i]
            if r>c: r,c = c,r
            # e_time_t[i,0] => vlen(t), e_time_t[i,1] => W
            w_map_t[(r,c)] = e_time_t[i,1]

    vlen_map_tp1 = {}
    if e_time_tplus is not None and row_tplus is not None:
        E_sp2 = len(row_tplus)
        for i in range(E_sp2):
            r = row_tplus[i]
            c = col_tplus[i]
            if r>c: r,c = c,r
            vlen_map_tp1[(r,c)] = e_time_tplus[i,0]

    ei_src = pyg_data_t.edge_index[0]
    ei_dst = pyg_data_t.edge_index[1]
    for e_i in range(E):
        s = ei_src[e_i].item()
        d = ei_dst[e_i].item()
        s_, d_ = (s,d) if s<d else (d,s)

        # vlen(t+1)
        if (s_,d_) in vlen_map_tp1:
            edge_feats[e_i,0] = vlen_map_tp1[(s_,d_)]
        else:
            edge_feats[e_i,0] = 0.0

        # W
        if (s_,d_) in w_map_t:
            edge_feats[e_i,1] = w_map_t[(s_,d_)]
        else:
            edge_feats[e_i,1] = 0.0

        # exists(t+1)
        if adjacency_tplus[s,d] == 1:
            edge_feats[e_i,2] = 1.0
        else:
            edge_feats[e_i,2] = 0.0

    label_dict = {
        "node_feats": node_feats,  # [N,12]
        "edge_feats": edge_feats,  # [E,3]
    }
    return label_dict

def fullgraph_for_dataset(all_datasets_path, out_dir):
    """
    1) Load 'all_datasets' => list of simulation runs, each a list of (spektral_graph_t, adjacency_{t+1})
    2) For each time t, convert G_t => PyG with shape:
         node feat => [N,12]
         edge feat => [E,3] => [vlen(t), W, exist(t)]
       Then build label_dict => [N,12], [E,3] => next-step [vlen, W, exist].
    3) Save (pyg_data_t, label_dict).
    """
    import torch
    os.makedirs(out_dir, exist_ok=True)

    with open(all_datasets_path, "rb") as f:
        all_datasets = pickle.load(f)

    print(f"Loaded all_datasets from '{all_datasets_path}'. Found {len(all_datasets)} sims.\n")

    for sim_idx, ds in enumerate(all_datasets):
        # ds = [ (G_t0, adj0plus), (G_t1, adj1plus), ... ]
        n_timepoints = len(ds)
        print(f"=== Simulation #{sim_idx+1}: {n_timepoints} timepoints. ===")

        for t_idx in range(n_timepoints - 1):
            (spektral_graph_t, adj_label_tplus) = ds[t_idx]
            if adj_label_tplus is None:
                continue

            # Next graph for node feats => G_{t+1}
            (spektral_graph_tplus, _) = ds[t_idx+1]

            # Convert G_t => PyG
            pyg_data_t = convert_spektral_to_pyg(spektral_graph_t)

            adjacency_tplus = adj_label_tplus   # shape [N, N]

            x_time_t       = spektral_graph_t.x       # [N,12]
            e_time_t       = spektral_graph_t.e       # [E2,2]
            row_t          = getattr(spektral_graph_t.a, "row", None)
            col_t          = getattr(spektral_graph_t.a, "col", None)

            x_time_tplus   = spektral_graph_tplus.x    # [N,12]
            e_time_tplus   = spektral_graph_tplus.e    # [E2,2]
            row_tplus      = getattr(spektral_graph_tplus.a, "row", None)
            col_tplus      = getattr(spektral_graph_tplus.a, "col", None)

            label_dict = build_fullgraph_label(
                pyg_data_t,
                adjacency_tplus,
                x_time_t, x_time_tplus,
                e_time_t, e_time_tplus,
                row_t, col_t,
                row_tplus, col_tplus
            )

            out_name = f"fullgraph_sim{sim_idx+1}_t{t_idx}.pkl"
            out_path = os.path.join(out_dir, out_name)
            with open(out_path, "wb") as f_out:
                pickle.dump((pyg_data_t, label_dict), f_out)

            print(f"   Saved [Sim={sim_idx+1}, t={t_idx}] => {out_path}")

        print("-----------------------------------------------------")

    print("Done building full-graph dataset with 3-edge-features. Each time step => (pyg_data, label_dict).")


In [37]:

# if __name__ == "__main__":

#     ALL_DATASETS_PATH = "all_datasets.pkl"  
#     # file you previously created that holds the list of datasets: 
#     #   all_datasets[sim_index] => list of (Graph, adjacency_label)

#     OUT_SUBGRAPHS_DIR = "subgraphs_output"  
#     # We'll store the neighbor-sampled subgraphs in this directory.

#     # If you want to specify neighbor-sampler parameters:
#     NUM_NEIGHBORS = [6, 6, 6]  # e.g. 3-layer sampling: 6 neighbors for the first node, then 6 neighbors per each of these 6 1st layer neighbors. If neighbor # < # defined then takes all neighbors. 
#     BATCH_SIZE = 1           # If 1, each subgraph is "centered" on a single node per iteration

#     neighbor_sampling_for_dataset(
#         all_datasets_path=ALL_DATASETS_PATH,   # path to 'all_datasets.pkl'
#         out_dir=OUT_SUBGRAPHS_DIR,
#         num_neighbors=NUM_NEIGHBORS, # length can be flexibly adjusted to increase or decrease layers
#         batch_size=BATCH_SIZE
#     )

#     print("Neighbor sampling complete.")

if __name__ == "__main__":
    data_dir = "all_datasets.pkl"  
    out_dir = "fullgraphs_output" 
    fullgraph_for_dataset(data_dir, out_dir)


Loaded all_datasets from 'all_datasets.pkl'. Found 10 sims.

=== Simulation #1: 11 timepoints. ===
   Saved [Sim=1, t=0] => fullgraphs_output/fullgraph_sim1_t0.pkl
   Saved [Sim=1, t=1] => fullgraphs_output/fullgraph_sim1_t1.pkl
   Saved [Sim=1, t=2] => fullgraphs_output/fullgraph_sim1_t2.pkl
   Saved [Sim=1, t=3] => fullgraphs_output/fullgraph_sim1_t3.pkl
   Saved [Sim=1, t=4] => fullgraphs_output/fullgraph_sim1_t4.pkl
   Saved [Sim=1, t=5] => fullgraphs_output/fullgraph_sim1_t5.pkl
   Saved [Sim=1, t=6] => fullgraphs_output/fullgraph_sim1_t6.pkl
   Saved [Sim=1, t=7] => fullgraphs_output/fullgraph_sim1_t7.pkl
   Saved [Sim=1, t=8] => fullgraphs_output/fullgraph_sim1_t8.pkl
   Saved [Sim=1, t=9] => fullgraphs_output/fullgraph_sim1_t9.pkl
-----------------------------------------------------
=== Simulation #2: 11 timepoints. ===
   Saved [Sim=2, t=0] => fullgraphs_output/fullgraph_sim2_t0.pkl
   Saved [Sim=2, t=1] => fullgraphs_output/fullgraph_sim2_t1.pkl
   Saved [Sim=2, t=2] => full

In [2]:
import os
import pickle
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data


###############################################################################
# 1) EdgeFeatureConv (unchanged)
###############################################################################
class EdgeFeatureConv(nn.Module):
    """
    A single message-passing layer that uses node embeddings + edge attributes.
    message_ij = MLP(h_i || h_j || e_ij)
    aggregated_j = sum_{i in N(j)} message_ij
    h_j_new = ReLU(aggregated_j)
    """

    def __init__(self, node_dim, edge_dim, hidden_dim):
        """
        :param node_dim: dimension of node embeddings
        :param edge_dim: dimension of edge features
        :param hidden_dim: dimension of output embeddings
        """
        super().__init__()
        in_dim = node_dim * 2 + edge_dim  # cat(h_i, h_j, e_ij)
        self.message_mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, x, edge_index, edge_attr):
        """
        :param x: shape [N, node_dim]  (node embeddings)
        :param edge_index: shape [2, E]
        :param edge_attr: shape [E, edge_dim]
        :return: x_out: [N, hidden_dim]
        """
        src = edge_index[0]  # [E]
        dst = edge_index[1]  # [E]

        h_src = x[src]       # [E, node_dim]
        h_dst = x[dst]       # [E, node_dim]

        msg_input = torch.cat([h_src, h_dst, edge_attr], dim=1)  # [E, node_dim*2 + edge_dim]
        messages = self.message_mlp(msg_input)                    # [E, hidden_dim]

        # sum over destination
        N = x.size(0)
        out = torch.zeros(N, messages.size(1), device=x.device)
        out.index_add_(0, dst, messages)  # sum into out[dst]

        return F.relu(out)


###############################################################################
# 2) GNCAFullGraphModel (unchanged)
###############################################################################
class GNCAFullGraphModel(nn.Module):
    """
    GNN-based GNCA with full in/out dimension matching:
      - Input node feats => 12
         (indices 0..3 dynamic, 4..11 static)
      - Input edge feats => 3
         (index 0 => vlen, index 1 => W, index 2 => exists(t))
      => Output node feats => 12
         We'll do partial MSE on [0..3].
      => Output edge feats => 3
         We'll do MSE on index0 => vlen,
         no penalty on index1 => W,
         BCE on index2 => adjacency logit
    """

    def __init__(
        self,
        node_in_dim=12,
        edge_in_dim=3,
        hidden_dim=64,
        num_layers=3,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.node_in_dim = node_in_dim
        self.edge_in_dim = edge_in_dim

        # 1) initial node embedding
        self.node_emb = nn.Linear(node_in_dim, hidden_dim)

        # 2) stack of EdgeFeatureConv layers
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(EdgeFeatureConv(hidden_dim, edge_in_dim, hidden_dim))

        # 3) final node head => 12 outputs
        self.node_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_in_dim)  # output 12
        )

        # 4) final edge head => 3 outputs
        #    index0 => next vlen
        #    index1 => next W
        #    index2 => adjacency logit
        self.edge_head = nn.Sequential(
            nn.Linear(hidden_dim*2 + edge_in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, edge_in_dim)  # 3
        )

    def forward(self, x, edge_index, edge_attr):
        """
        :param x: shape [N, 12]
        :param edge_index: [2, E]
        :param edge_attr: [E, 3]
        :return: (node_out, edge_out)
          node_out => shape [N, 12]
          edge_out => shape [E, 3]
        """
        # 1) embed
        h = self.node_emb(x)
        h = F.relu(h)

        # 2) pass through EdgeFeatureConv layers
        for conv in self.convs:
            h = conv(h, edge_index, edge_attr)

        # 3) node output => shape [N,12]
        node_out = self.node_head(h)

        # 4) edge output => shape [E,3]
        src, dst = edge_index
        h_src = h[src]
        h_dst = h[dst]
        edge_in = torch.cat([h_src, h_dst, edge_attr], dim=1)  # [E, hidden_dim*2 + 3]
        edge_out = self.edge_head(edge_in)

        return node_out, edge_out


###############################################################################
# 3) The training function WITHOUT shuffling across simulations/time
###############################################################################
def gnca_train_fullgraph(
    model,
    seq_dataset_dict,
    device=torch.device("cpu"),
    lr=1e-3,
    epochs=20,
):
    """
    :param model: GNCAFullGraphModel
    :param seq_dataset_dict: dict { sim_id: list of (t, pyg_data, label_dict) in ascending time }
    :param device: CPU or CUDA
    :param lr: learning rate
    :param epochs: # training epochs

    We do:
     - For epoch in range(epochs):
       - For each simulation sim_id in ascending order:
         - For each time step (t) in ascending order:
           - Train on that (pyg_data, label_dict)
    So no random shuffle across sims or times.
    """
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        print(f"\n===== EPOCH {epoch+1}/{epochs} =====")
        total_loss = 0.0
        total_samples = 0

        # For each sim_id in sorted order
        for sim_id in sorted(seq_dataset_dict.keys()):
            # For each time in ascending order
            sim_data_list = seq_dataset_dict[sim_id]
            
            # sim_data_list is sorted by t
            for (t, pyg_data, label_dict) in sim_data_list:
                pyg_data = pyg_data.to(device)
                print(f"  Training on sim_id={sim_id}, time={t}")
                node_target = torch.tensor(label_dict["node_feats"], dtype=torch.float, device=device)
                edge_target = torch.tensor(label_dict["edge_feats"], dtype=torch.float, device=device)

                optimizer.zero_grad()

                node_out, edge_out = model(
                    pyg_data.x, 
                    pyg_data.edge_index, 
                    pyg_data.edge_attr
                )
                # node_out => [N,12]
                # edge_out => [E,3], with index => 0 => vlen, 1 => W, 2 => adjacency logit

                # 1) Node MSE => only indices [0..3]
                node_mse = F.mse_loss(node_out[:,0:4], node_target[:,0:4])

                # 2) Edge adjacency => BCE => index2
                adj_logit = edge_out[:,2]
                adj_label = edge_target[:,2]
                edge_bce = F.binary_cross_entropy_with_logits(adj_logit, adj_label)

                # 3) Edge vlen => MSE => index0
                edge_vlen_pred = edge_out[:,0]
                edge_vlen_targ = edge_target[:,0]
                edge_mse = F.mse_loss(edge_vlen_pred, edge_vlen_targ)

                loss = node_mse + edge_bce + edge_mse
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                total_samples += 1

        avg_loss = total_loss / total_samples
        print(f"Epoch {epoch+1}: mean loss across all sims/time = {avg_loss:.4f}")


###############################################################################
# 4) Save Model Function
###############################################################################
def save_model(model, save_path="gnca_model.pth"):
    """
    Saves the trained model to a file.

    :param model: Trained PyTorch model.
    :param save_path: File path to save the model (default: 'gnca_model.pth').
    """
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")


###############################################################################
# 5) Dataset Parsing Function (Unchanged)
###############################################################################
def build_sequential_dataset(data_dir):
    all_files = [f for f in os.listdir(data_dir) if f.endswith(".pkl")]
    seq_dataset = {}
    for fname in all_files:
        base = os.path.splitext(fname)[0]
        leftover = base.replace("fullgraph_sim", "")
        parts = leftover.split("_t")
        sim_id = int(parts[0])
        time_id = int(parts[1])
        path = os.path.join(data_dir, fname)
        with open(path, "rb") as f:
            pyg_data, label_dict = pickle.load(f)
        if sim_id not in seq_dataset:
            seq_dataset[sim_id] = []
        seq_dataset[sim_id].append((time_id, pyg_data, label_dict))
    for sim_id in seq_dataset.keys():
        seq_dataset[sim_id].sort(key=lambda x: x[0])
    return seq_dataset


###############################################################################
# 6) Main Training Function (Modified to Save Model)
###############################################################################
def main_train_gnca(data_dir="fullgraphs_output", num_layers=3, hidden_dim=64, epochs=20, lr=1e-3, save_path="gnca_model.pth"):
    """
    Main function to train GNCA and save the model.

    :param data_dir: Directory containing .pkl files (default: "fullgraphs_output").
    :param num_layers: Number of GNN layers (default: 3).
    :param hidden_dim: Hidden dimension size (default: 64).
    :param epochs: Number of training epochs (default: 20).
    :param lr: Learning rate for Adam optimizer (default: 1e-3).
    :param save_path: File path to save the trained model (default: "gnca_model.pth").
    """

    # 1) Build sequential dataset dictionary
    seq_dataset_dict = build_sequential_dataset(data_dir)
    if not seq_dataset_dict:
        print("No data found in", data_dir)
        return

    first_sim = sorted(seq_dataset_dict.keys())[0]
    first_t, first_pyg_data, first_label = seq_dataset_dict[first_sim][0]
    print(f"First sim_id={first_sim}, time={first_t}")
    print("Sample node features shape:", first_pyg_data.x.shape)
    print("Sample edge features shape:", first_pyg_data.edge_attr.shape)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 2) Build model
    model = GNCAFullGraphModel(node_in_dim=12, edge_in_dim=3, hidden_dim=hidden_dim, num_layers=num_layers)

    # 3) Train model
    gnca_train_fullgraph(model, seq_dataset_dict, device=device, lr=lr, epochs=epochs)

    # 4) Save trained model
    save_model(model, save_path=save_path)

    print("Training complete! The model has been saved.")


# Run this function in a Jupyter Notebook cell:
main_train_gnca(data_dir="fullgraphs_output", save_path="gnca_model.pth")

First sim_id=1, time=0
Sample node features shape: torch.Size([840, 12])
Sample edge features shape: torch.Size([5040, 3])

===== EPOCH 1/20 =====
  Training on sim_id=1, time=0
  Training on sim_id=1, time=1
  Training on sim_id=1, time=2
  Training on sim_id=1, time=3
  Training on sim_id=1, time=4
  Training on sim_id=1, time=5
  Training on sim_id=1, time=6
  Training on sim_id=1, time=7
  Training on sim_id=1, time=8
  Training on sim_id=1, time=9
  Training on sim_id=2, time=0
  Training on sim_id=2, time=1
  Training on sim_id=2, time=2
  Training on sim_id=2, time=3
  Training on sim_id=2, time=4
  Training on sim_id=2, time=5
  Training on sim_id=2, time=6
  Training on sim_id=2, time=7
  Training on sim_id=2, time=8
  Training on sim_id=2, time=9
  Training on sim_id=3, time=0
  Training on sim_id=3, time=1
  Training on sim_id=3, time=2
  Training on sim_id=3, time=3
  Training on sim_id=3, time=4
  Training on sim_id=3, time=5
  Training on sim_id=3, time=6
  Training on si

## Prediction

In [48]:
# import os
# import numpy as np
# import torch
# import torch.nn.functional as F
# from torch_geometric.data import Data

# def predict_gnca_time_evolution(
#     model,
#     init_npy: str,
#     out_dir: str,
#     t_end: int = 10,
#     t_int: int = 1,
#     threshold: float = 0.5,
# ):
#     """
#     Uses a trained GNCA model to evolve from time=0 up to time=t_end in increments of t_int.
#     1) Loads data_0.npy => builds an initial Spektral Graph => convert to PyG with
#        node feats = [N,12], edge feats = [E,3].
#     2) For each step from 0..(t_end-1) in steps of t_int:
#        - forward pass => node_out, edge_out
#        - threshold edge_out[:,2] => next-step adjacency
#        - update node & edge features
#        - store them or save them if needed
#     :param model: a GNCAFullGraphModel (already loaded)
#     :param init_npy: path to data_0.npy (the dictionary with 'cell_x','cell_type','area','perimeter','cell_adj', etc.)
#     :param out_dir: where to optionally save the predicted steps
#     :param t_end: final time index (like 100)
#     :param t_int: time step increments
#     :param threshold: adjacency logit threshold
#     """
#     os.makedirs(out_dir, exist_ok=True)

#     # 1) Load data_0.npy
#     data_init = np.load(init_npy, allow_pickle=True).item()
#     cell_x    = data_init["cell_x"]      # shape [n_c, 2]
#     cell_type = data_init["cell_type"]   # shape [n_c]
#     area      = data_init["area"]        # shape [n_c]
#     perimeter = data_init["perimeter"]   # shape [n_c]
#     cell_adj  = data_init["cell_adj"]    # shape [n_c,n_c]

#     n_c = cell_x.shape[0]

#     # 2) Build the node feature array => shape [N,12]
#     #    [ x, y, area, perimeter, cell_type, A0, P0, Dr, kappaA, kappaP, a, k ]
#     #    We'll assume you have e.g. param file if needed or you can store them directly.
#     #    For now, let's store placeholders for the static fields [4..11].
#     #    We do an example approach with some constant placeholders if not provided:
#     #    But presumably you have them from the dictionary or param file. 
#     #    We'll do an example with 'k' = data_init, etc.  If not, just fill your constants.

#     # Example placeholders for each cell i
#     # We'll copy them from data_init if you have them. 
#     # But let's do a quick approach:
#     if "k" in data_init:
#         k_ = data_init["k"]  # single float?
#     else:
#         k_ = 2.5  # example

#     # If your data_init had "A0" => shape [2], we might do a quick approach:
#     # But for simplicity, let's do placeholders or skip. We'll do 0.0 for the static ones:
#     node_feats = []
#     for i in range(n_c):
#         x_i, y_i = cell_x[i]
#         # dynamic
#         row = [
#             x_i, y_i, area[i], perimeter[i]
#         ]
#         # example static placeholders 
#         row += [float(cell_type[i]), 0.9, 3.812, 50.0, 0.4, 0.07, 0.25, k_]
#         node_feats.append(row)
#     node_feats = np.array(node_feats, dtype=np.float32)  # shape [N,12]

#     # 3) Build edge array => shape [E,2] => [vlen, W], for unique edges. Then duplicate.
#     #    We set 'exists(t)=1' in the final PyG step, so your "convert" function does that.
#     rows, cols, edge_f = [], [], []
#     for i in range(n_c):
#         for j in range(i+1, n_c):
#             if cell_adj[i,j] == 1:
#                 rows.append(i)
#                 cols.append(j)
#                 # example vlen=0.0, W=0.08 or something
#                 vlen_ij = 0.0
#                 W_ij    = 0.08
#                 edge_f.append([vlen_ij, W_ij])

#     # Duplicate for directed
#     rows_d = rows + cols
#     cols_d = cols + rows
#     edge_f_d = edge_f + edge_f
#     row_ar = np.array(rows_d, dtype=np.int64)
#     col_ar = np.array(cols_d, dtype=np.int64)
#     edge_f_ar = np.array(edge_f_d, dtype=np.float32)

#     # 4) Build the initial PyG data
#     import scipy.sparse
#     data_ar = np.ones_like(row_ar, dtype=np.float32)
#     A_coo = scipy.sparse.coo_matrix((data_ar, (row_ar, col_ar)), shape=(n_c,n_c))

#     # Build a "Spektral" Graph placeholder for conversion
#     from spektral.data import Graph as SpektralGraph
#     G_init = SpektralGraph(
#         x=node_feats,
#         a=A_coo,
#         e=edge_f_ar
#     )

#     # Convert to PyG with your "convert_spektral_to_pyg"
#     pyg_data_0 = convert_spektral_to_pyg(G_init)
#     # shape node => [N,12], edge => [E,3], with index2 => exist(t)=1

#     # 5) Iterative forward passes
#     device = next(model.parameters()).device
#     # We store the trajectory in memory
#     trajectory = []
#     # time = 0 => the initial
#     trajectory.append( (0, pyg_data_0) )

#     current_pyg_data = pyg_data_0.clone()

#     # We'll do times => 0+ t_int, 0+2 t_int, ... up to <= t_end
#     steps = list(range(0, t_end+1, t_int))
#     # The first step is 0, which we already have, so we'll skip or handle in loop
#     # We'll do for i in steps[1:] => do the update
#     for step_t in steps[1:]:
#         # forward
#         current_pyg_data = current_pyg_data.to(device)
#         node_out, edge_out = model(
#             current_pyg_data.x,
#             current_pyg_data.edge_index,
#             current_pyg_data.edge_attr
#         )

#         # 6) Build next node/edge for time step step_t
#         # node_out => shape [N,12]
#         # edge_out => shape [E,3], index0 => vlen, index1 => W, index2 => adjacency logit

#         # adjacency
#         adj_logit = edge_out[:,2]
#         adj_prob = torch.sigmoid(adj_logit).detach().cpu().numpy()
#         # threshold
#         adj_exists = (adj_prob > threshold).astype(np.float32)

#         # We'll produce new edge_attr => shape [E,3]
#         # next vlen = edge_out[:,0].detach().cpu().numpy()
#         # next W    = edge_out[:,1] or keep old W, up to you
#         next_vlen = edge_out[:,0].detach().cpu().numpy()
#         next_W    = edge_out[:,1].detach().cpu().numpy()

#         new_edge_attr = []
#         E = edge_out.shape[0]
#         for e_i in range(E):
#             new_edge_attr.append([
#                 next_vlen[e_i],
#                 next_W[e_i],
#                 adj_exists[e_i]
#             ])
#         new_edge_attr = np.array(new_edge_attr, dtype=np.float32)

#         # new node => shape [N,12]
#         new_node = node_out.detach().cpu().numpy()

#         # 7) Build new adjacency from adj_exists => if 0 => remove edge
#         # We do that by "disabling" edges with adjacency=0 => optional
#         # or we keep them but set them = 0 for exist(t).
#         # For the "convert" logic to re-run, you'd need a new Spektral graph. We'll do it all in PyG.

#         # We'll just keep the same edge_index, but skip edges where exist=0 => effectively no "active" link
#         # If you truly want to remove them, you can mask them out from edge_index, but that changes the
#         # shape of edge_out. We'll keep the shape, just store exist=0 => no effect in next pass if your message passing
#         # uses exist(t) as a multiplier. You might do so.

#         new_edge_attr_t = torch.FloatTensor(new_edge_attr)

#         new_node_t = torch.FloatTensor(new_node)
#         # Build new PyG data
#         new_pyg_data = Data(
#             x=new_node_t,
#             edge_index=current_pyg_data.edge_index.clone(),
#             edge_attr=new_edge_attr_t
#         )

#         # store in trajectory
#         trajectory.append( (step_t, new_pyg_data) )

#         # update current
#         current_pyg_data = new_pyg_data

#     # 8) Optionally save each step to disk
#     for (tt, pydata) in trajectory:
#         save_path = os.path.join(out_dir, f"pred_time{tt}.pkl")
#         # We'll just pickle the node/edge arrays
#         node_arr = pydata.x.numpy()
#         edge_arr = pydata.edge_attr.numpy()
#         with open(save_path, "wb") as f:
#             pickle.dump((node_arr, edge_arr), f)
#         print(f"Saved predicted step t={tt} to {save_path}")

#     return trajectory


In [6]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import argparse
import pickle
import scipy.sparse
from spektral.data import Graph as SpektralGraph
from torch_geometric.data import Data

###############################################################################
# 1) load_params function (unchanged)
###############################################################################
def load_params(py_path):
    import importlib.util
    print(f"Loading parameters from: {py_path}")
    spec = importlib.util.spec_from_file_location("param_module", py_path)
    param_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(param_module)

    p = {
        "v0": getattr(param_module, "v0", None),
        "W": getattr(param_module, "W", None),
        "A0": getattr(param_module, "A0", None),
        "P0": getattr(param_module, "P0", None),
        "Dr": getattr(param_module, "Dr", None),
        "kappa_A": getattr(param_module, "kappa_A", None),
        "kappa_P": getattr(param_module, "kappa_P", None),
        "a": getattr(param_module, "a", None),
        "k": getattr(param_module, "k", None),
    }
    print("Loaded params:", p)
    return p

###############################################################################
# 2) convert_spektral_to_pyg (unchanged)
###############################################################################
def convert_spektral_to_pyg(spektral_graph):
    """
    We take node feats shape [N,12], edge feats shape [E,2], 
    produce a PyG Data with:
      x => shape [N,12]
      edge_attr => shape [E,3], 
        index => 0 => vlen, 1 => W, 2 => exist(t)=1
    """
    x_np = spektral_graph.x  # [N,12]
    row_s = spektral_graph.a.row
    col_s = spektral_graph.a.col
    edge_index_np = np.vstack([row_s, col_s])  # [2, E]

    import torch
    edge_index_t = torch.LongTensor(edge_index_np)

    if spektral_graph.e is not None:
        old_e = spektral_graph.e  # shape [E,2] => [vlen, W]
        E = old_e.shape[0]
        new_e = np.zeros((E, 3), dtype=np.float32)
        new_e[:,0:2] = old_e[:,0:2]   # copy (vlen, W)
        new_e[:,2]   = 1.0           # exist(t)=1
        edge_attr_t = torch.FloatTensor(new_e)
    else:
        # If no e => build all exist=1, vlen=0, W=0
        E = edge_index_np.shape[1]
        new_e = np.zeros((E,3), dtype=np.float32)
        new_e[:,2] = 1.0
        edge_attr_t = torch.FloatTensor(new_e)

    from torch_geometric.data import Data
    pyg_data = Data(
        x=torch.FloatTensor(x_np),
        edge_index=edge_index_t,
        edge_attr=edge_attr_t
    )
    return pyg_data

###############################################################################
# 3) Main "predict" function
###############################################################################
def predict_gnca_evolution(
    model,
    param_path: str,
    data_dir: str,
    t_start: int,
    t_end: int,
    t_int: int,
    out_dir: str,
    threshold: float=0.5
):
    """
    1) Loads param file => build node feats for time t_start from data_{t_start}.npy
    2) Builds an initial Spektral Graph => convert to PyG => forward pass => produce next step
    3) Repeats until t_end in increments of t_int
    4) Saves each predicted step as .pkl
    5) Every 10 steps => print summary of node[0..3] & edge[0] stats
    6) Also convert each step to a .npy with the same structure as data_0.npy
    """
    os.makedirs(out_dir, exist_ok=True)

    # 1) Load param dictionary
    p = load_params(param_path)

    # 2) load data_{t_start}.npy => build node feats & edge feats
    file_0 = os.path.join(data_dir, f"data_{t_start}.npy")
    if not os.path.exists(file_0):
        print(f"ERROR: cannot find {file_0} for initial time {t_start}")
        return

    data0 = np.load(file_0, allow_pickle=True).item()

    cell_x = data0["cell_x"]            # shape [n_c, 2]
    cell_type = data0["cell_type"]      # shape [n_c]
    area = data0["area"]               # shape [n_c]
    perimeter = data0["perimeter"]      # shape [n_c]
    cell_adj = data0["cell_adj"]        # shape [n_c, n_c]
    
    # ### PLACEHOLDER FOR VLEN REPLACED ###
    # We now parse 'edge_voronoi_length' from the dictionary, if it exists:
    edge_voronoi_length = data0.get("edge_voronoi_length", None)
    if edge_voronoi_length is not None:
        print("> Found 'edge_voronoi_length' in data dict. Will use real vlen.")
    else:
        print("> No 'edge_voronoi_length' found. Will fallback to 0.0 for vlen.")
    # ### end replacement ###

    n_c = cell_x.shape[0]

    # Build node feats => shape [N,12]
    node_feats = []
    for i in range(n_c):
        ctype = cell_type[i]
        x_i, y_i = cell_x[i]
        A0_val = p["A0"][ctype]
        P0_val = p["P0"][ctype]
        row = [x_i, y_i, area[i], perimeter[i]]  # dynamic
        row += [
            float(ctype),
            A0_val,
            P0_val,
            p["Dr"],
            p["kappa_A"],
            p["kappa_P"],
            p["a"],
            p["k"],
        ]
        node_feats.append(row)
    node_feats = np.array(node_feats, dtype=np.float32)

    # Build unique edges => shape [E,2], then duplicate
    rows, cols, e_feats = [], [], []
    for i in range(n_c):
        for j in range(i+1, n_c):
            if cell_adj[i,j]==1:
                rows.append(i)
                cols.append(j)
                ctype_i = cell_type[i]
                ctype_j = cell_type[j]
                Wij = p["W"][ctype_i][ctype_j]
                
                # ### PLACEHOLDER REMOVED ###
                # old: e_feats.append([0.0, Wij])
                # now parse real vlen if available:
                if edge_voronoi_length is not None:
                    vlen_ij = edge_voronoi_length[i, j]
                else:
                    vlen_ij = 0.0

                e_feats.append([vlen_ij, Wij])

    row_d = rows + cols
    col_d = cols + rows
    e_feats_d = e_feats + e_feats
    row_ar = np.array(row_d, dtype=np.int64)
    col_ar = np.array(col_d, dtype=np.int64)
    e_feats_ar = np.array(e_feats_d, dtype=np.float32)

    import scipy.sparse
    data_ar = np.ones_like(row_ar, dtype=np.float32)
    a_coo = scipy.sparse.coo_matrix((data_ar, (row_ar, col_ar)), shape=(n_c,n_c))

    # Build Spektral Graph
    G0 = SpektralGraph(
        x=node_feats,
        a=a_coo,
        e=e_feats_ar
    )

    # Convert => PyG => shape [N,12], [E,3 => (vlen, W, exist=1)]
    pyg_data_0 = convert_spektral_to_pyg(G0)

    device = next(model.parameters()).device
    current_data = pyg_data_0.clone()
    current_time = t_start

    # store the trajectory as (time, PyG Data)
    trajectory = []
    trajectory.append((current_time, current_data))

    # times => t_start + t_int.. up to t_end
    steps = list(range(current_time + t_int, t_end+1, t_int))

    # original cell_type from time0 for re-use in output .npy
    original_cell_type = cell_type.copy()

    for idx, t_next in enumerate(steps, start=1):
        # forward pass
        current_data = current_data.to(device)
        node_out, edge_out = model(
            current_data.x,
            current_data.edge_index,
            current_data.edge_attr
        )
        # node_out => [N,12]
        # edge_out => [E,3], index => 0 => vlen, 1 => W, 2 => adjacency logit
        logit = edge_out[:,2]
        prob = torch.sigmoid(logit).detach().cpu().numpy()
        exist_next = (prob > threshold).astype(np.float32)

        vlen_next = edge_out[:,0].detach().cpu().numpy()
        W_next    = edge_out[:,1].detach().cpu().numpy()

        E = edge_out.shape[0]
        new_edge_attr = []
        for e_i in range(E):
            new_edge_attr.append([
                vlen_next[e_i],
                W_next[e_i],
                exist_next[e_i]
            ])
        new_edge_attr = np.array(new_edge_attr, dtype=np.float32)

        new_node = node_out.detach().cpu().numpy()

        from torch_geometric.data import Data as PyGData
        new_node_t = torch.FloatTensor(new_node)
        new_edge_t = torch.FloatTensor(new_edge_attr)
        new_pyg_data = PyGData(
            x=new_node_t,
            edge_index=current_data.edge_index.clone(),
            edge_attr=new_edge_t
        )

        trajectory.append((t_next, new_pyg_data))
        current_data = new_pyg_data

    # Save each step
    os.makedirs(out_dir, exist_ok=True)

    # Also step 0 is in trajectory[0], so let's include that in the output as well
    # We'll iterate over the entire trajectory, printing every 10 steps, etc.
    for idx, (tt, pydata) in enumerate(trajectory):
        node_arr = pydata.x.numpy()   # shape [N,12]
        edge_arr = pydata.edge_attr.numpy()  # shape [E,3]
        # E => same # edges as in duplication

        # 1) Save .pkl
        save_path_pkl = os.path.join(out_dir, f"pred_time{tt}.pkl")
        with open(save_path_pkl, "wb") as f:
            pickle.dump((node_arr, edge_arr), f)

        # 2) For interpretability => every 10 steps => print summary
        if (idx % 10) == 0:
            node_dyn = node_arr[:,0:4] # x,y,area,peri
            node_min = node_dyn.min(axis=0)
            node_max = node_dyn.max(axis=0)
            node_mean= node_dyn.mean(axis=0)
            print(f"[Step {tt}] Node dyn: min={node_min}, max={node_max}, mean={node_mean}")

            vlen_val = edge_arr[:,0]
            e_min, e_max, e_mean = vlen_val.min(), vlen_val.max(), vlen_val.mean()
            print(f"[Step {tt}] Edge vlen: min={e_min:.4f}, max={e_max:.4f}, mean={e_mean:.4f}")

        # 3) Build a new adjacency from exist=1
        n_c2 = node_arr.shape[0]
        adjacency_mat = np.zeros((n_c2,n_c2), dtype=np.int8)
        e_index = pydata.edge_index.numpy()  # shape[2, E]
        for e_i in range(e_index.shape[1]):
            r_ = e_index[0,e_i]
            c_ = e_index[1,e_i]
            if edge_arr[e_i,2] > 0.5:
                adjacency_mat[r_, c_] = 1
                adjacency_mat[c_, r_] = 1

        # 4) Save as data_{tt}.npy
        out_dict = {
            "cell_x": node_arr[:,0:2],
            "cell_type": original_cell_type,  # or node_arr[:,4]
            "area": node_arr[:,2],
            "perimeter": node_arr[:,3],
            "cell_adj": adjacency_mat
        }
        save_path_npy = os.path.join(out_dir, f"data_{tt}.npy")
        np.save(save_path_npy, out_dict, allow_pickle=True)

    print("Prediction complete. Steps stored in", out_dir)

###############################################################################
# CLI entry point
###############################################################################
def main_predict_gnca(
    model,
    param_path="params.py",
    data_dir="data",
    t_start=0,
    t_end=10,
    t_int=1,
    out_dir="prediction_output",
    threshold=0.5
):
    """
    Jupyter-compatible main function to predict GNCA evolution.

    :param model: Loaded PyTorch GNCA model.
    :param param_path: Path to param .py file
    :param data_dir: Directory with `data_{t}.npy`.
    :param t_start: Start time
    :param t_end: End time
    :param t_int: increment
    :param out_dir: Folder for output
    :param threshold: adjacency threshold
    """

    # model in eval mode
    model.eval()

    predict_gnca_evolution(
        model=model,
        param_path=param_path,
        data_dir=data_dir,
        t_start=t_start,
        t_end=t_end,
        t_int=t_int,
        out_dir=out_dir,
        threshold=threshold
    )

    print("\nPrediction complete. Steps stored in:", out_dir)



###############################################################################
# CLI entry point
###############################################################################
def main_predict_gnca(
    model,
    param_path="params.py",
    data_dir="data",
    t_start=0,
    t_end=10,
    t_int=1,
    out_dir="prediction_output",
    threshold=0.5
):
    """
    Jupyter-compatible main function to predict GNCA evolution.

    :param model: Loaded PyTorch GNCA model.
    :param param_path: Path to param .py file
    :param data_dir: Directory with `data_{t}.npy`.
    :param t_start: Start time
    :param t_end: End time
    :param t_int: increment
    :param out_dir: Folder for output
    :param threshold: adjacency threshold
    """

    # model in eval mode
    model.eval()

    predict_gnca_evolution(
        model=model,
        param_path=param_path,
        data_dir=data_dir,
        t_start=t_start,
        t_end=t_end,
        t_int=t_int,
        out_dir=out_dir,
        threshold=threshold
    )

    print("\nPrediction complete. Steps stored in:", out_dir)

###############################################################################
# Main to run via CLI
###############################################################################
def main_predict_gnca(
    model,
    param_path="params.py",
    data_dir="data",
    t_start=0,
    t_end=10,
    t_int=1,
    out_dir="prediction_output",
    threshold=0.5
):
    """
    Jupyter-compatible main function to predict GNCA evolution.

    :param model: Loaded PyTorch GNCA model.
    :param param_path: Path to the param .py file (e.g., "1.py").
    :param data_dir: Directory with `data_{t}.npy` files.
    :param t_start: Start time for prediction.
    :param t_end: End time for prediction.
    :param t_int: Time step increment.
    :param out_dir: Directory to save predictions.
    :param threshold: Adjacency threshold for edge_out[:,2].
    """

    # Ensure model is in eval mode
    model.eval()

    # Run prediction
    predict_gnca_evolution(
        model=model,
        param_path=param_path,
        data_dir="/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/validation_Fig5I/parameters_DPAC1_test",
        t_start=t_start,
        t_end=t_end,
        t_int=t_int,
        out_dir=out_dir,
        threshold=threshold
    )

    print("\nPrediction complete. Steps stored in:", out_dir)


# Example usage in Jupyter Notebook:
# Load trained model

# 1) Recreate the model with the same structure
model = GNCAFullGraphModel(
    node_in_dim=12,
    edge_in_dim=3,
    hidden_dim=64,
    num_layers=3
)

# 2) Load the state dict
model.load_state_dict(torch.load("gnca_model.pth", map_location="cpu"))
model.eval()


#Run prediction
main_predict_gnca(
    model=model,
    param_path="/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/parameters_DPAC1.py",
    data_dir="/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/validation_Fig5I/parameters_DPAC1_test",
    t_start=0,
    t_end=2000,
    t_int=100,
    out_dir="GNCA_prediction_output"
)


/var/folders/_2/xkrjp2q52cn8mxmjrvp731500000gq/T/ipykernel_69883/1893344464.py:438: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("gnca_mode

Loading parameters from: /Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/parameters_DPAC1.py
Loaded params: {'v0': [0.0, 1.5], 'W': ([0.0, 0.1], [0.1, 0.0]), 'A0': [0.9, 0.9], 'P0': [3.812, 3.812], 'Dr': 40, 'kappa_A': 0.3, 'kappa_P': 0.05, 'a': 0.2, 'k': 2}
> Found 'edge_voronoi_length' in data dict. Will use real vlen.
[Step 0] Node dyn: min=[8.6742721e-04 3.0069461e-04 9.8574674e-01 3.9684324e+00], max=[59.99916   13.999672   1.0144806  4.0155854], mean=[29.856558    7.02265     0.99999994  3.9936695 ]
[Step 0] Edge vlen: min=0.0001, max=59.0123, mean=1.2738
[Step 1000] Node dyn: min=[78.25382   24.1427     1.183738   5.3014283], max=[13372.342    2354.5303    461.39685  1013.976  ], mean=[3479.12     603.6624   130.27165  188.54399]
[Step 1000] Edge vlen: min=-292.1089, max=1239.6349, mean=201.9372
[Step 2000] Node dyn: min=[12496.831    1947.2589    460.50275   575.9188 ], max=[4310084.5   776630.5   158599.25  326634.2 ], mean=[1234999.5    213609.8     46525.344